# Generate Training Data for Model Router Fine-Tuning

This notebook generates training data in the format required by the [Model Router fine-tuning notebook](./finetuning_model_router_rest.ipynb).

## Workflow
1. **Configure** – Set endpoint, LLM list, judge model, and paths.
2. **Load prompts** – Read prompts from a simple text/JSONL file.
3. **Generate responses** – For each prompt, call all LLMs in parallel via the model-router deployment.
4. **Judge responses** – Use a judge LLM to score each model's response (binary: 1 = good, 0 = bad).
5. **Write training data** – Output JSONL with `messages`, `metrics`, and `usage` fields.

## Prerequisites
- A deployed **Model Router** endpoint (used to route requests to individual LLMs via headers).
- Python packages: `requests`.
- A file containing prompts (one per line or JSONL with a `prompt` field).

## 0. Imports

In [1]:
import json
import os
import sys

sys.path.insert(0, os.path.abspath("."))

from src.data_generation_helpers import (
    process_prompts,
)

## 1. Configuration

Set all parameters below. This is the **only cell** you need to edit.

In [ ]:
# ── Azure AI Foundry Project Endpoint ─────────────────────────────────────
RESOURCE_NAME = "https://modelrouter-eus2.cognitiveservices.azure.com"  # Replace with your actual endpoint
#ENDPOINT = "https://<YOUR_RESOURCE_NAME>.services.ai.azure.com/api/projects/<YOUR_PROJECT_NAME>"
#API_KEY  = "<YOUR_API_KEY>"  # Replace with your actual API key
API_KEY = ''
# ── Model Router Deployment ───────────────────────────────────────────────
# The deployment name of your model-router (used to route to individual LLMs)
#DEPLOYMENT_NAME = "<YOUR_MODEL_ROUTER_DEPLOYMENT_NAME>"
DEPLOYMENT_NAME = "model-router"  # Replace with your actual deployment name

# ── LLM List ──────────────────────────────────────────────────────────────
# Each LLM is identified by name and version.
# See supported models: https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/model-router#supported-models
LLM_LIST = [
    {"name": "gpt-4.1", "version": "2025-04-14"},
    {"name": "gpt-4.1-mini", "version": "2025-04-14"},
    {"name": "gpt-5-mini", "version": "2025-08-07"},
    {"name": "Deepseek-V3.1", "version": "1"},
    {"name": "Llama-4-Maverick-17B-128E-Instruct-FP8", "version": "1"},
]

# ── Judge Model ───────────────────────────────────────────────────────────
# The judge LLM evaluates responses and assigns binary scores.
JUDGE_MODEL_NAME    = "gpt-5"  # Use a strong model as judge
JUDGE_MODEL_VERSION = "2025-08-07"

# ── Judge Prompt Template ─────────────────────────────────────────────────
# Must contain {prompt} and {responses} placeholders.
# Output must be a JSON object mapping model names to binary scores (1 or 0).
JUDGE_PROMPT_TEMPLATE = """You are an impartial judge evaluating AI model responses.

## Original Prompt
{prompt}

## Model Responses
{responses}

## Instructions
Evaluate each model's response for correctness, helpfulness, and quality.
Assign a binary score: 1 = good/correct, 0 = bad/incorrect.

Return ONLY a JSON object mapping model names to scores. Example:
{{"gpt-4.1": 1, "gpt-4.1-mini": 1, "gpt-4.1-nano": 0}}
"""

# ── Input / Output Paths ──────────────────────────────────────────────────
PROMPTS_FILE = "data/prompts.txt"  # One prompt per line (or .jsonl with "prompt" field)
OUTPUT_FILE  = "data/generated_training_data.jsonl"

# ── Parallelism ───────────────────────────────────────────────────────────
MAX_WORKERS = len(LLM_LIST)  # One thread per LLM for maximum parallelism

## 2. Load Prompts

Load prompts from the input file. Supports:
- **Plain text** (`.txt`): One prompt per line.
- **JSONL** (`.jsonl`): Each line is a JSON object with a `"prompt"` field.

In [3]:
prompts = []

with open(PROMPTS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        if PROMPTS_FILE.endswith(".jsonl"):
            record = json.loads(line)
            prompts.append(record["prompt"])
        else:
            prompts.append(line)

print(f"Loaded {len(prompts)} prompts from {PROMPTS_FILE}")
print(f"\nFirst 5 prompts:")
for idx, p in enumerate(prompts[:5]):
    print(f"  {idx + 1}. {p[:150]}")

Loaded 200 prompts from data/prompts.txt

First 5 prompts:
  1. Answer the following multiple choice question. The last line of your response should be of the following format: 'Answer: $LETTER' (without quotes) wh
  2. Answer the following question. The last line of your response should be of the form: 'Answer: $EXPRESSION' (without quotes) where EXPRESSION is a sequ
  3. Answer the following yes/no question. The last line of your response should be one of: 'Answer: Yes' or 'Answer: No' (without quotes). Think step by s
  4. Answer the following plausibility question. The last line of your response should be one of: 'Answer: Yes' or 'Answer: No' (without quotes). Think ste
  5. Answer the following yes/no question. The last line of your response should be one of: 'Answer: Yes' or 'Answer: No' (without quotes). Think step by s


## 3. Generate Training Data

For each prompt:
1. Call all LLMs in **parallel** via the model-router deployment (using routing headers).
2. Send all responses to the **judge LLM** to generate binary metrics.
3. Combine into the training record format: `{messages, metrics, usage}`.

In [4]:
output_path = process_prompts(
    endpoint=RESOURCE_NAME,
    api_key=API_KEY,
    deployment_name=DEPLOYMENT_NAME,
    prompts=prompts,
    llm_list=LLM_LIST,
    judge_model_name=JUDGE_MODEL_NAME,
    judge_model_version=JUDGE_MODEL_VERSION,
    judge_prompt_template=JUDGE_PROMPT_TEMPLATE,
    max_workers=MAX_WORKERS,
    output_path=OUTPUT_FILE,
)

Processing prompt 1/200...
Error calling Deepseek-V3.1: list index out of range
Processing prompt 2/200...
Processing prompt 3/200...
Processing prompt 4/200...
Processing prompt 5/200...
Error calling gpt-4.1-mini: list index out of range
Error calling gpt-5-mini: list index out of range
Processing prompt 6/200...
Processing prompt 7/200...
Processing prompt 8/200...
Processing prompt 9/200...
Processing prompt 10/200...
Processing prompt 11/200...
Processing prompt 12/200...
Processing prompt 13/200...
Processing prompt 14/200...


IndexError: list index out of range

## 4. Preview Generated Data

Inspect the first few records to verify the format matches what the fine-tuning notebook expects.

In [ ]:
print("=== Output Format ===")
print("Each line is a JSON object with: messages, metrics, usage\n")

with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        record = json.loads(line)
        print(f"--- Record {i + 1} ---")
        print(json.dumps(record, indent=2)[:1000])
        print()

## Next Steps

The generated file at `OUTPUT_FILE` is ready to be used as training data in the
[fine-tuning notebook](./finetuning_model_router_rest.ipynb).

Set `TRAINING_FILE_PATH` in the fine-tuning notebook to point to the generated file:
```python
TRAINING_FILE_PATH = "data/generated_training_data.jsonl"
```